# **Project Name** — Voyage Analytics — Historic-Label Classification

##### **Project Type** — EDA / Classification
##### **Contribution** — Individual
##### **Team Member 1 — Harsh Paul**

# **Project Summary**

This notebook demonstrates a classification workflow for the binary
historical label supplied in users.csv. It covers data quality, preprocessing,
holdout evaluation, and responsible-use documentation.

This standalone notebook uses the supplied travel_capstone data, follows the
provided EDA submission template, and is ready to run in Google Colab. It explains
the data checks, modelling choices, evaluation approach, and business meaning of
the findings rather than only printing code output.

# **GitHub Link**

[https://github.com/harshpaul1999-tech/Voyage-Analytics](https://github.com/harshpaul1999-tech/Voyage-Analytics)

# **Problem Statement**

#### **Define Your Business Objective?**

Demonstrate a technically reproducible classification pipeline on the
supplied historical label. It is an educational exercise only and must not be used
to infer a person’s gender identity or make decisions about people.

All model inputs are limited to information available at prediction time. The
holdout logic is kept separate from fitting so reported results reflect unseen
examples rather than memorisation.

# **General Guidelines**

- Run every cell from top to bottom in a fresh Colab runtime.
- The setup cell clones the public GitHub repository automatically in Colab.
- The charts and metrics are calculated from the supplied data, not typed in.
- A fixed random seed is used when a split or model needs randomness.

# ***Let's Begin !***

## ***1. Know Your Data***

### Import Libraries

In [ ]:
from pathlib import Path
import sys
if "google.colab" in sys.modules:
    if not Path("/content/Voyage-Analytics").exists():
        !git clone --depth 1 https://github.com/harshpaul1999-tech/Voyage-Analytics.git /content/Voyage-Analytics
    %cd /content/Voyage-Analytics

import warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, f1_score, mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVC

sns.set_theme(style="whitegrid", palette="deep")
RANDOM_STATE = 42
DATA_DIR = Path("data/raw")

### Dataset Loading

In [ ]:
users = pd.read_csv(DATA_DIR / "users.csv")
for col in ['company', 'name', 'gender']:
    users[col] = users[col].astype(str).str.strip()
print("Loaded users:", users.shape)

### Dataset First View

In [ ]:
display(users.head())

### Dataset Rows & Columns count

In [ ]:
print(users.shape)

### Dataset Information

In [ ]:
users.info()

#### Duplicate Values

In [ ]:
print('Duplicate rows:', users.duplicated().sum())

#### Missing Values/Null Values

In [ ]:
display(users.isna().sum().sort_values(ascending=False).to_frame('missing_values'))

### What did you know about your dataset?

The dataset records traveller information. The historic label is used only
as a supplied technical target; it must not be interpreted as a person’s identity
or used as a production decision variable.

## ***2. Understanding Your Variables***

### Variables Description

| Variable | Meaning | Model role |
|---|---|---|
| code | User identifier | Excluded identifier |
| company | Associated organisation | Exploratory context |
| name | Traveller name | Educational text feature |
| age | Traveller age | Educational numeric feature |
| gender | Historic binary dataset label | Technical target only |

### Check Unique Values for each variable.

In [ ]:
display(users.nunique().sort_values(ascending=False).to_frame('unique_values'))

## ***3. Data Wrangling***

### Data Wrangling Code

In [ ]:
class_df = users[users['gender'].str.lower().isin(['male', 'female'])].copy()
class_df = class_df.dropna(subset=['name', 'age', 'gender'])
class_df['first_name'] = class_df['name'].str.split().str[0].str.lower()
class_df['gender'] = class_df['gender'].str.lower()
class_df['name_length'] = class_df['first_name'].str.len()
print("Rows retained:", len(class_df))

### What all manipulations have you done and insights you found?

The code uses only rows matching the two historical labels present in the data,
normalises first names, and removes incomplete rows. These operations improve
technical consistency, but they do not make names and ages appropriate for
real-world identity inference.

## ***4. Data Visualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

#### Chart - 1: Age distribution

##### 1. Why did you pick the specific chart?
A histogram checks the scale of the numeric feature.

##### 2. What is/are the insight(s) found from the chart?
It shows the age coverage of the supplied users.

##### 3. Will the gained insights help creating a positive business impact?
It validates input quality before standardisation.

In [ ]:
sns.histplot(data=class_df, x='age', bins=30, kde=True); plt.title('Age distribution'); plt.show()

#### Chart - 2: Historical label balance

##### 1. Why did you pick the specific chart?
A countplot reveals target balance.

##### 2. What is/are the insight(s) found from the chart?
It indicates whether a majority class could distort accuracy.

##### 3. Will the gained insights help creating a positive business impact?
It motivates stratification and weighted F1.

In [ ]:
sns.countplot(data=class_df, x='gender'); plt.title('Historic label balance'); plt.show()

#### Chart - 3: Age by historical label

##### 1. Why did you pick the specific chart?
A boxplot compares distributions without implying causality.

##### 2. What is/are the insight(s) found from the chart?
It shows overlap between the supplied groups.

##### 3. Will the gained insights help creating a positive business impact?
It supports cautious discussion of weak demographic signals.

In [ ]:
sns.boxplot(data=class_df, x='gender', y='age'); plt.title('Age by historic label'); plt.show()

#### Chart - 4: Top traveller companies

##### 1. Why did you pick the specific chart?
Company provides dataset-composition context.

##### 2. What is/are the insight(s) found from the chart?
It shows which organisations supply the most rows.

##### 3. Will the gained insights help creating a positive business impact?
It flags sample concentration when considering generalisation.

In [ ]:
class_df['company'].value_counts().head(12).sort_values().plot(kind='barh', title='Top companies'); plt.show()

#### Chart - 5: First-name length

##### 1. Why did you pick the specific chart?
Name length is a text-quality check.

##### 2. What is/are the insight(s) found from the chart?
It finds empty, unusually short, or unusually long name tokens.

##### 3. Will the gained insights help creating a positive business impact?
It reduces avoidable TF–IDF errors.

In [ ]:
sns.histplot(data=class_df, x='name_length', bins=20); plt.title('First-name length'); plt.show()

#### Chart - 6: Age density by label

##### 1. Why did you pick the specific chart?
Overlaid densities show distribution overlap.

##### 2. What is/are the insight(s) found from the chart?
It prevents overstating age as a separation rule.

##### 3. Will the gained insights help creating a positive business impact?
It reinforces the educational-only boundary.

In [ ]:
sns.histplot(data=class_df, x='age', hue='gender', stat='density', common_norm=False, element='step'); plt.title('Age density by historic label'); plt.show()

#### Chart - 7: Most frequent initials

##### 1. Why did you pick the specific chart?
Initials summarize text composition without listing names.

##### 2. What is/are the insight(s) found from the chart?
It shows recurring character patterns.

##### 3. Will the gained insights help creating a positive business impact?
It explains why character n-grams can fit historical labels.

In [ ]:
class_df['first_name'].str[0].value_counts().head(15).sort_values().plot(kind='barh', title='Frequent first initials'); plt.show()

#### Chart - 8: Rows by age band

##### 1. Why did you pick the specific chart?
Bands make coverage easier to inspect.

##### 2. What is/are the insight(s) found from the chart?
It identifies sparse age regions.

##### 3. Will the gained insights help creating a positive business impact?
Sparse segments can be monitored in future technical validation.

In [ ]:
bands = pd.cut(class_df['age'], [0,20,30,40,50,60,120], right=False); sns.countplot(y=bands, order=bands.value_counts().index.sort_values()); plt.title('Rows by age band'); plt.show()

#### Chart - 9: Company-label composition

##### 1. Why did you pick the specific chart?
A grouped chart checks sample composition.

##### 2. What is/are the insight(s) found from the chart?
It visualizes historic labels in the largest companies.

##### 3. Will the gained insights help creating a positive business impact?
It surfaces possible sample bias rather than implying a company effect.

In [ ]:
top = class_df['company'].value_counts().head(8).index; sns.countplot(data=class_df[class_df['company'].isin(top)], y='company', hue='gender', order=top); plt.title('Historic label composition'); plt.show()

#### Chart - 10: Missing-value audit

##### 1. Why did you pick the specific chart?
A bar chart makes completeness auditable.

##### 2. What is/are the insight(s) found from the chart?
It confirms required rows were cleaned before fitting.

##### 3. Will the gained insights help creating a positive business impact?
It supports reliable future retraining.

In [ ]:
class_df.isna().sum().plot(kind='bar', title='Missing values after cleaning'); plt.show()

## **5. Solution to Business Objective**

The technical workflow is complete: validated inputs, train/test split, pipeline,
metrics, artifact tracking, and a guarded API. The business conclusion is not to
deploy identity inference; this remains a documented educational demonstration.

## ***6. Feature Engineering & Data Pre-processing***

Character-level TF–IDF extracts spelling patterns from normalised first names. Age
is standardised. Both transformations are inside the Pipeline, so they are learned
from training data only and applied consistently to the holdout set.

## ***7. Model Implementation***

### Character n-gram TF–IDF + Linear SVC

In [ ]:
X = class_df[['first_name', 'age']].rename(columns={'first_name': 'name'})
y = class_df['gender']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=.20, random_state=RANDOM_STATE, stratify=y
)
historic_label_model = Pipeline([
    ('features', ColumnTransformer([
        ('name_ngrams', TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 5),
                                        sublinear_tf=True), 'name'),
        ('age', StandardScaler(), ['age']),
    ])),
    ('classifier', LinearSVC(C=.8, class_weight='balanced')),
])
historic_label_model.fit(X_train, y_train)
predicted_label = historic_label_model.predict(X_test)

## ***8. Model Performance & Evaluation***

Accuracy measures overall agreement with the held-out historical labels. Weighted
F1 balances precision and recall while accounting for label frequency. Neither
metric measures identity or suitability for real-world use.

In [ ]:
metrics = {
    'accuracy': float(accuracy_score(y_test, predicted_label)),
    'weighted_f1': float(f1_score(y_test, predicted_label, average='weighted')),
    'train_rows': len(X_train), 'test_rows': len(X_test),
}
display(pd.Series(metrics).to_frame('value'))
print(classification_report(y_test, predicted_label))

### Evaluation interpretation

The split is stratified to retain the historic label ratio. The score quantifies
only agreement with that dataset field. It must not be described as a model of a
person’s gender identity or used for eligibility, employment, safety, ranking, or
personalisation.

## ***9. MLOps & Deployment***

The repository saves the model with an explicit responsible-use notice, records
metrics in JSON and MLflow, and returns the same warning from its Flask endpoint.
This demonstrates that ethical scope belongs in artifacts and interfaces, not just
in oral explanation.

# **Conclusion**

This notebook demonstrates a reproducible classification lifecycle while stating
its non-production boundary. The responsible recommendation is to retain it as a
classroom dataset-label exercise.

### ***Hurrah! You have successfully completed your Classification Capstone Project !!!***